# Retrain without training trips 33 and 54

Exclude `00000033` and `00000054` from model fitting and CV. Retain the raw files and previous results. Reuse the same saved trip folds, selected hyperparameters, feature list and candidate settings. No new tuning is performed.

The ten student-test trips stay identical. They have already been used for development comparisons, so this is a repeated development evaluation, not a new independent test.

Removing difficult validation trips changes the CV population. A higher CV accuracy by itself is not evidence that predictions improved; compare the unchanged student-test set separately.

In [1]:
%run 06_model_helpers.ipynb
import xgboost as xgb
from time import perf_counter
from datetime import datetime
from zoneinfo import ZoneInfo

excluded = ["00000033", "00000054"]
output = Path("results/exclude_33_54")
output.mkdir(parents=True, exist_ok=True)
params = json.loads(Path("outputs/step7_xgb_sample10000/best_params.json").read_text())["params"]
n_jobs = 8
data, points = load_model_data()
original_points = len(points)
data = data[~data["trip_id"].isin(excluded)].reset_index(drop=True)
points = points[~points["trip_id"].isin(excluded)].copy()
assert data["trip_id"].nunique() == 78 and set(excluded).isdisjoint(data["trip_id"])
assert data["group_id"].is_monotonic_increasing
config = {"excluded_trip_ids": excluded, "remaining_trips": 78, "training_points": len(points),
          "removed_labelled_points": original_points - len(points), "candidate_rows": len(data),
          "params": params, "fold_file": contract["fold_file"], "seed": seed,
          "candidate_settings": contract["candidate_settings"]}
(output / "config.json").write_text(json.dumps(config, indent=2) + "\n")
print(config, flush=True)

{'excluded_trip_ids': ['00000033', '00000054'], 'remaining_trips': 78, 'training_points': 182635, 'removed_labelled_points': 23697, 'candidate_rows': 9202147, 'params': {'max_depth': 6, 'learning_rate': 0.29992371455654276, 'n_estimators': 50, 'min_child_weight': 2.968888733614375, 'subsample': 0.7653830020461954, 'colsample_bytree': 0.6828369642189636, 'reg_lambda': 0.1627321975366077, 'gamma': 1.0947188593873698}, 'fold_file': 'outputs/folds/group_folds.csv', 'seed': 42, 'candidate_settings': {'radius_m': 210, 'K': 100, 'target_miss_rate': 0.001, 'train_miss_rate': 0.0037027702925382395, 'labelled_points': 206332}}


In [2]:
def fit_ranker(frame, params):
    sizes = frame.groupby("group_id")["label"].agg(["size", "sum"])
    eligible = sizes.index[(sizes["size"] >= 2) & (sizes["sum"] == 1)]
    training = frame[frame["group_id"].isin(eligible)]
    assert training["group_id"].is_monotonic_increasing
    groups = training.groupby("group_id", sort=False).size().to_numpy()
    model = xgb.XGBRanker(objective="rank:ndcg", eval_metric="ndcg@1", tree_method="hist",
                          random_state=seed, n_jobs=n_jobs, **params)
    model.fit(training[feature_columns], training["label"], group=groups, verbose=False)
    return model


## Retrain and score the unchanged student-test set first

Fit all eligible groups from the remaining 78 training trips. Predict before attaching the separate reference labels. Keep candidate misses in the Top-1 denominator. Time model prediction plus point-wise softmax on precomputed features.

In [3]:
start = perf_counter()
model = fit_ranker(data, params)
final_fit_s = perf_counter() - start
model.save_model(output / "model.json")
test = pd.read_parquet("outputs/student_test/features.parquet")
assert test["label"].isna().all()
nearest = test["distance"] - test["distance_above_nearest_m"]
test["distance_ratio"] = ((test["distance"] + 1) / (nearest + 1)).astype("float32")
start = perf_counter()
raw = model.predict(test[feature_columns])
probability = point_softmax(raw, test["group_id"])
predict_s = perf_counter() - start
predictions = test[["group_id", "segment_id"]].copy()
predictions["score"], predictions["raw_score"] = probability, raw
predictions.to_csv(output / "student_test_predictions.csv", index=False)

test_points = pd.read_csv("outputs/student_test/points.csv", dtype={"trip_id": str})
test_points = test_points[~test_points["bad_gps"]].copy()
valid = test["group_id"].isin(test_points["group_id"])
scoring = test.loc[valid, ["group_id", "segment_id"]].copy()
truth = scoring["group_id"].map(test_points.set_index("group_id")["true_arc_id"])
scoring["label"] = scoring["segment_id"].eq(truth).astype("int8")
metrics = score_points(scoring, probability[valid.to_numpy()], test_points)
metrics["final_fit_s"], metrics["prediction_s"] = final_fit_s, predict_s
(output / "student_test_metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
previous = json.loads(Path("results/step7_xgb/tuned_sample10000/student_test_metrics.json").read_text())
comparison = pd.DataFrame([{"training": "80 trips", **previous}, {"training": "78 trips: exclude 33,54", **metrics}])
comparison.to_csv(output / "student_test_comparison.csv", index=False)
print("Student-test accuracy:", metrics["top1"], "| Change (percentage points):", 100 * (metrics["top1"] - previous["top1"]), flush=True)
display(comparison[["training", "top1", "macro_f1", "log_loss", "points", "candidate_misses"]])
del test, scoring, raw, probability, predictions

Student-test accuracy: 0.8943110647181628 | Change (percentage points): 0.2818371607515635


,training,top1,macro_f1,log_loss,points,candidate_misses
0,80 trips,0.891493,0.944856,0.310644,19160,0
1,"78 trips: exclude 33,54",0.894311,0.946288,0.296880,19160,0


## CV on the remaining trips

Keep each surviving trip in its original saved fold. Refit on the other surviving trips for each fold. This measures performance on a smaller training/validation population than the original CV and must be labelled accordingly.

In [4]:
records = []
oof_probability = np.full(len(data), np.nan, dtype=np.float64)
for fold in range(5):
    training = data[data["fold"] != fold]
    validation = data[data["fold"] == fold]
    valid_points = points[points["fold"] == fold]
    assert set(training["trip_id"]).isdisjoint(validation["trip_id"])
    assert set(excluded).isdisjoint(training["trip_id"])
    start = perf_counter()
    fold_model = fit_ranker(training, params)
    fit_s = perf_counter() - start
    start = perf_counter()
    raw = fold_model.predict(validation[feature_columns])
    probability = point_softmax(raw, validation["group_id"])
    elapsed = perf_counter() - start
    oof_probability[validation.index] = probability
    records.append({"fold": fold, **score_points(validation, probability, valid_points), "fit_s": fit_s, "predict_s": elapsed})
    pd.DataFrame(records).to_csv(output / "cv_metrics.csv", index=False)
    print("Completed fold", fold, "| Top-1:", records[-1]["top1"], flush=True)

assert np.isfinite(oof_probability).all()
pooled = score_points(data, oof_probability, points)
(output / "pooled_metrics.json").write_text(json.dumps(pooled, indent=2) + "\n")
oof = data[["group_id", "segment_id", "fold"]].copy()
oof["score"] = oof_probability
oof.to_csv(output / "oof_xgb_ranker.csv", index=False)
print("Pooled OOF accuracy on remaining 78 trips:", pooled["top1"], flush=True)

Completed fold 0 | Top-1: 0.854239101364001


Completed fold 1 | Top-1: 0.8572181874835215


Completed fold 2 | Top-1: 0.9225866736948032


Completed fold 3 | Top-1: 0.9346493448062538


Completed fold 4 | Top-1: 0.9001724096063719


Pooled OOF accuracy on remaining 78 trips: 0.8965532345935883


## Record the experiment

Save exclusion provenance and append the run to the experiment ledger. The previously selected export is retained; this notebook reports the exclusion experiment separately.

In [5]:
timestamp = datetime.now(ZoneInfo("Asia/Kolkata"))
cv = pd.DataFrame(records)
row = {"run_id": "exclude_33_54_" + timestamp.strftime("%Y%m%d_%H%M%S"), "timestamp": timestamp.isoformat(),
       "model": "xgb_ranker", "params": json.dumps(params), "seed": seed,
       "excluded_trips": json.dumps(excluded), "training_trips": 78,
       "fold_scheme": contract["fold_scheme"] + "; omit trips 33 and 54 without changing remaining assignments",
       "radius_m": contract["candidate_settings"]["radius_m"], "K": contract["candidate_settings"]["K"],
       "final_fit_s": final_fit_s, "final_predict_s": predict_s, "output_dir": str(output),
       "test_status": "repeated student-test development comparison; not a fresh holdout"}
for metric in ["top1", "macro_f1", "precision", "recall", "log_loss", "candidate_coverage"]:
    row[f"cv_{metric}_mean"], row[f"cv_{metric}_std"] = cv[metric].mean(), cv[metric].std()
    row[f"pooled_{metric}"], row[f"test_{metric}"] = pooled[metric], metrics[metric]
ledger = Path("results/experiments.csv")
existing = pd.read_csv(ledger) if ledger.exists() else pd.DataFrame()
pd.concat([existing, pd.DataFrame([row])], ignore_index=True).to_csv(ledger, index=False)
print("Saved retrained model, predictions, metrics, and experiment record.")

Saved retrained model, predictions, metrics, and experiment record.
